# NLP Disaster Tweets — Ensemble v3: DistilBERT + BERTweet + RoBERTa

Поверх `ensemble_v2.ipynb` (public LB 0.84094). Добавляем третью модель `roberta-base` для увеличения диверсификации ансамбля.

**Зачем третья модель:**
- DistilBERT — small 6-layer, общий English pretrain
- BERTweet — RoBERTa-arch, 850M tweets pretrain
- **RoBERTa** — large 12-layer, общий English pretrain, **другой токенизатор и другая нормализация** vs остальные

Diminishing returns: третья модель обычно даёт +0.003-0.007.

**Оптимизация:** если в директории уже лежат `distil_oof_v2.npy`, `distil_test_v2.npy`, `bert_oof_v2.npy`, `bert_test_v2.npy` от прошлого `ensemble_v2.ipynb`, ноутбук их подгружает и не перепрогоняет — только RoBERTa обучается заново.

**Пайплайн:**
1. Воспроизводим cleaning из v2 (для согласованности artifacts)
2. Препроцессинг под три модели
3. DistilBERT, BERTweet — загружаем из артефактов (или обучаем если нет)
4. RoBERTa — обучаем 5-fold
5. Усреднение probs (1/3 каждой) → threshold tuning → overlap fix → submission

## 1. Импорты и настройки

In [ ]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from tqdm.auto import tqdm

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.backends.mps.is_available():
    DEVICE = torch.device('mps');  NUM_GPUS = 1
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda'); NUM_GPUS = torch.cuda.device_count()
else:
    DEVICE = torch.device('cpu');  NUM_GPUS = 0

NUM_WORKERS = 4 if DEVICE.type == 'cuda' else 0
PIN_MEMORY  = DEVICE.type == 'cuda'

print(f'Device: {DEVICE} | GPUs: {NUM_GPUS} | PyTorch: {torch.__version__}')

## 2. Загрузка + воспроизведение cleaning (как в ensemble_v2)

Используем те же OOFs от `ensemble.ipynb` для воспроизведения cleaning — это гарантирует совместимость с уже сохранёнными `*_v2.npy` артефактами.

In [ ]:
df_train = pd.read_csv('data/train.csv')
df_test  = pd.read_csv('data/test.csv')

assert os.path.exists('distil_oof.npy') and os.path.exists('bert_oof.npy'), \
    'Нужны OOFs от ensemble.ipynb для cleaning. Запусти его сначала.'

distil_oof_old = np.load('distil_oof.npy')
bert_oof_old   = np.load('bert_oof.npy')
ensemble_oof_old = (distil_oof_old + bert_oof_old) / 2

def majority_label(labels):
    counts = labels.value_counts()
    if len(counts) > 1 and counts.iloc[0] == counts.iloc[1]:
        return 1
    return counts.idxmax()

resolved = df_train.groupby('text', sort=False)['target'].agg(majority_label).reset_index()
first_meta = df_train.groupby('text', sort=False)[['keyword', 'location']].first().reset_index()
df_train_dedup = resolved.merge(first_meta, on='text')

first_pos = df_train.reset_index().groupby('text', sort=False)['index'].first()
dedup_oof = ensemble_oof_old[first_pos.loc[df_train_dedup['text']].values]

FLIP_HIGH, FLIP_LOW = 0.90, 0.10
y_dedup = df_train_dedup['target'].values.copy()
y_dedup[(dedup_oof > FLIP_HIGH) & (y_dedup == 0)] = 1
y_dedup[(dedup_oof < FLIP_LOW)  & (y_dedup == 1)] = 0

df_train_clean = df_train_dedup.copy()
df_train_clean['target'] = y_dedup
y_clean = y_dedup

print(f'Cleaned train: {len(df_train_clean)} строк')

## 3. Препроцессинг под три модели

Разный preprocessing → больше диверсификации в ансамбле:
- **DistilBERT** — агрессивная очистка, lowercase, только text
- **BERTweet** — `HTTPURL`/`@USER` caps, эмодзи, text + keyword + location
- **RoBERTa** — `http`/`@user` lowercase tokens, case сохранён, text + keyword + location

In [ ]:
def clean_for_distilbert(text: str) -> str:
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    text = re.sub(r'[^\w\s.,!?\'-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def normalize_for_bertweet(text: str) -> str:
    text = re.sub(r'http\S+|www\S+', 'HTTPURL', text)
    text = re.sub(r'@\w+', '@USER', text)
    text = re.sub(r'&amp;', '&', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def normalize_for_roberta(text: str) -> str:
    text = re.sub(r'http\S+|www\S+', 'http', text)
    text = re.sub(r'@\w+', '@user', text)
    text = re.sub(r'&amp;', '&', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def build_with_meta(row, text_col) -> str:
    keyword  = str(row['keyword']).replace('%20', ' ') if pd.notna(row['keyword'])  else ''
    location = str(row['location'])                    if pd.notna(row['location']) else ''
    parts = [p for p in [keyword, location, row[text_col]] if p]
    return ' | '.join(parts)

# DistilBERT — только text
df_train_clean['text_distilbert'] = df_train_clean['text'].apply(clean_for_distilbert)
df_test['text_distilbert']        = df_test['text'].apply(clean_for_distilbert)

# BERTweet — нормализация + конкатенация
df_train_clean['_bert_clean'] = df_train_clean['text'].apply(normalize_for_bertweet)
df_test['_bert_clean']        = df_test['text'].apply(normalize_for_bertweet)
df_train_clean['text_bertweet'] = df_train_clean.apply(lambda r: build_with_meta(r, '_bert_clean'), axis=1)
df_test['text_bertweet']        = df_test.apply(lambda r: build_with_meta(r, '_bert_clean'), axis=1)

# RoBERTa — другая нормализация + конкатенация
df_train_clean['_rob_clean'] = df_train_clean['text'].apply(normalize_for_roberta)
df_test['_rob_clean']        = df_test['text'].apply(normalize_for_roberta)
df_train_clean['text_roberta'] = df_train_clean.apply(lambda r: build_with_meta(r, '_rob_clean'), axis=1)
df_test['text_roberta']        = df_test.apply(lambda r: build_with_meta(r, '_rob_clean'), axis=1)

print('Примеры входов трёх моделей:')
print(f'DistilBERT : {df_train_clean["text_distilbert"].iloc[0]}')
print(f'BERTweet   : {df_train_clean["text_bertweet"].iloc[0]}')
print(f'RoBERTa    : {df_train_clean["text_roberta"].iloc[0]}')

## 4. KFold-функция и Dataset

In [ ]:
MAX_LEN       = 128
EPOCHS        = 3
LEARNING_RATE = 2e-5
N_SPLITS      = 5
BATCH_SIZE    = 32 * max(NUM_GPUS, 1)


class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, labels=None):
        self.encodings = tokenizer(
            list(texts), truncation=True, padding='max_length',
            max_length=MAX_LEN, return_tensors='pt'
        )
        self.labels = labels.reset_index(drop=True) if labels is not None else None

    def __len__(self):
        return self.encodings['input_ids'].shape[0]

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels.iloc[idx], dtype=torch.long)
        return item


def predict_probs(model, loader, has_labels=True):
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model(**batch)
            probs.extend(torch.softmax(out.logits, dim=1)[:, 1].cpu().numpy())
            if has_labels:
                labels.extend(batch['labels'].cpu().numpy())
    return np.array(probs), (np.array(labels) if has_labels else None)


def train_kfold(model_name, train_texts, test_texts, y, tokenizer, tag=''):
    test_ds = TextDataset(pd.Series(test_texts), tokenizer)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE,
                             num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    oof_probs  = np.zeros(len(train_texts))
    test_probs = np.zeros(len(test_texts))
    fold_scores = []

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(train_texts, y), 1):
        print(f'\n--- [{tag}] Fold {fold}/{N_SPLITS} ---')

        tr_ds = TextDataset(pd.Series(train_texts[tr_idx]), tokenizer, pd.Series(y[tr_idx]))
        va_ds = TextDataset(pd.Series(train_texts[va_idx]), tokenizer, pd.Series(y[va_idx]))

        tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
        va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE,
                               num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
        model = model.to(DEVICE)
        if NUM_GPUS > 1:
            model = nn.DataParallel(model)
        base = model.module if isinstance(model, nn.DataParallel) else model

        optimizer = torch.optim.AdamW(base.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
        total_steps = EPOCHS * len(tr_loader)
        scheduler = get_linear_schedule_with_warmup(optimizer, total_steps // 10, total_steps)

        val_p, val_y = None, None
        for ep in range(1, EPOCHS + 1):
            model.train()
            for batch in tqdm(tr_loader, desc=f'  train ep{ep}', leave=False):
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                out = model(**batch)
                loss = out.loss.mean()
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(base.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
            val_p, val_y = predict_probs(model, va_loader)
            print(f'  ep{ep} val F1@0.5: {f1_score(val_y, (val_p > 0.5).astype(int)):.4f}')

        oof_probs[va_idx] = val_p
        fold_scores.append(f1_score(val_y, (val_p > 0.5).astype(int)))

        test_p, _ = predict_probs(model, test_loader, has_labels=False)
        test_probs += test_p / N_SPLITS

        del model, base, optimizer, scheduler
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

    print(f'\n[{tag}] Mean fold F1@0.5: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}')
    print(f'[{tag}] Overall OOF F1@0.5: {f1_score(y, (oof_probs > 0.5).astype(int)):.4f}')
    return oof_probs, test_probs, fold_scores

## 5. DistilBERT (reuse артефактов v2 если есть)

In [ ]:
if os.path.exists('distil_oof_v2.npy') and os.path.exists('distil_test_v2.npy'):
    distil_oof  = np.load('distil_oof_v2.npy')
    distil_test = np.load('distil_test_v2.npy')
    assert len(distil_oof) == len(y_clean), 'distil_oof_v2 size mismatch — пере-прогон нужен'
    print(f'✓ Loaded DistilBERT artifacts (OOF F1@0.5: {f1_score(y_clean, (distil_oof > 0.5).astype(int)):.4f})')
else:
    print('Артефакты DistilBERT не найдены — обучаем заново.')
    distil_tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
    distil_oof, distil_test, _ = train_kfold(
        'distilbert-base-uncased',
        df_train_clean['text_distilbert'].values,
        df_test['text_distilbert'].values,
        y_clean, distil_tokenizer, tag='DistilBERT'
    )
    np.save('distil_oof_v2.npy', distil_oof)
    np.save('distil_test_v2.npy', distil_test)

## 6. BERTweet (reuse артефактов v2 если есть)

In [ ]:
if os.path.exists('bert_oof_v2.npy') and os.path.exists('bert_test_v2.npy'):
    bert_oof  = np.load('bert_oof_v2.npy')
    bert_test = np.load('bert_test_v2.npy')
    assert len(bert_oof) == len(y_clean), 'bert_oof_v2 size mismatch — пере-прогон нужен'
    print(f'✓ Loaded BERTweet artifacts (OOF F1@0.5: {f1_score(y_clean, (bert_oof > 0.5).astype(int)):.4f})')
else:
    print('Артефакты BERTweet не найдены — обучаем заново.')
    bert_tokenizer = AutoTokenizer.from_pretrained('vinai/bertweet-base', use_fast=False, normalization=False)
    bert_oof, bert_test, _ = train_kfold(
        'vinai/bertweet-base',
        df_train_clean['text_bertweet'].values,
        df_test['text_bertweet'].values,
        y_clean, bert_tokenizer, tag='BERTweet'
    )
    np.save('bert_oof_v2.npy', bert_oof)
    np.save('bert_test_v2.npy', bert_test)

## 7. RoBERTa (новая модель)

In [ ]:
if os.path.exists('roberta_oof_v3.npy') and os.path.exists('roberta_test_v3.npy'):
    roberta_oof  = np.load('roberta_oof_v3.npy')
    roberta_test = np.load('roberta_test_v3.npy')
    print(f'✓ Loaded RoBERTa artifacts (OOF F1@0.5: {f1_score(y_clean, (roberta_oof > 0.5).astype(int)):.4f})')
else:
    roberta_tokenizer = AutoTokenizer.from_pretrained('FacebookAI/roberta-base')
    roberta_oof, roberta_test, _ = train_kfold(
        'FacebookAI/roberta-base',
        df_train_clean['text_roberta'].values,
        df_test['text_roberta'].values,
        y_clean, roberta_tokenizer, tag='RoBERTa'
    )
    np.save('roberta_oof_v3.npy',  roberta_oof)
    np.save('roberta_test_v3.npy', roberta_test)

## 8. Ensemble трёх моделей + threshold tuning

In [ ]:
oof_ensemble  = (distil_oof  + bert_oof  + roberta_oof)  / 3
test_ensemble = (distil_test + bert_test + roberta_test) / 3

print('--- OOF F1@0.5 ---')
print(f'DistilBERT       : {f1_score(y_clean, (distil_oof    > 0.5).astype(int)):.4f}')
print(f'BERTweet         : {f1_score(y_clean, (bert_oof      > 0.5).astype(int)):.4f}')
print(f'RoBERTa          : {f1_score(y_clean, (roberta_oof   > 0.5).astype(int)):.4f}')
print(f'2-model ensemble : {f1_score(y_clean, ((distil_oof + bert_oof)/2 > 0.5).astype(int)):.4f}')
print(f'3-model ensemble : {f1_score(y_clean, (oof_ensemble > 0.5).astype(int)):.4f}')

thresholds = np.arange(0.30, 0.61, 0.01)
f1_per_t = [f1_score(y_clean, (oof_ensemble > t).astype(int)) for t in thresholds]
best_idx = int(np.argmax(f1_per_t))
best_threshold = thresholds[best_idx]
best_oof_f1 = f1_per_t[best_idx]

print(f'\nBest 3-model threshold: {best_threshold:.2f}  →  OOF F1 = {best_oof_f1:.4f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds, f1_per_t, marker='o', markersize=3)
ax.axvline(best_threshold, color='red', linestyle='--', label=f'best={best_threshold:.2f}')
ax.set_xlabel('Threshold'); ax.set_ylabel('OOF F1 (3-model ensemble)')
ax.set_title('Ensemble v3: F1 vs threshold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 9. Classification report + CM

In [ ]:
oof_preds = (oof_ensemble > best_threshold).astype(int)
print(classification_report(y_clean, oof_preds, target_names=['Not Disaster', 'Disaster']))

cm = confusion_matrix(y_clean, oof_preds)
ConfusionMatrixDisplay(cm, display_labels=['Not Disaster', 'Disaster']).plot(cmap='Blues')
plt.title(f'Ensemble v3 OOF CM (F1={best_oof_f1:.4f}, t={best_threshold:.2f})')
plt.show()

## 10. Submission с overlap fix

In [ ]:
test_preds = (test_ensemble > best_threshold).astype(int).tolist()
print(f'До overlap fix → Disaster: {sum(test_preds)}, Not disaster: {len(test_preds) - sum(test_preds)}')

overlap_labels = (
    df_train_clean[df_train_clean['text'].isin(df_test['text'])]
    .set_index('text')['target']
    .to_dict()
)

overridden = 0
for i, text in enumerate(df_test['text']):
    if text in overlap_labels:
        test_preds[i] = int(overlap_labels[text])
        overridden += 1

print(f'Overlap fix: перезаписано {overridden} предсказаний')
print(f'После overlap fix → Disaster: {sum(test_preds)}, Not disaster: {len(test_preds) - sum(test_preds)}')

submission = pd.read_csv('data/sample_submission.csv')
submission['target'] = test_preds
submission.to_csv('submission_ensemble_v3.csv', index=False)

print('\nsubmission_ensemble_v3.csv saved')
submission.head()